# QuantumEdge - GIC 2026 Phase 3 Submission Verification

This verifier checks both scientific artifacts and explicit Phase 3 compliance requirements. It does not convert missing mandatory evidence into a warning. A final submission should report **0 FAIL**.


In [6]:
from pathlib import Path
import pandas as pd

print("Current working directory:")
print(Path.cwd().resolve())

failed = verification.loc[
    verification["Status"].eq("FAIL"),
    ["Category", "Item", "Detail"],
].reset_index(drop=True)

print(f"\nFailures found: {len(failed)}")
display(failed)

Current working directory:
/home/jovyan/phase 3 third run

Failures found: 1


,Category,Item,Detail
0,README,Working Launch on qBraid button,public repository URL required


In [7]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
rows = []


def record(category, item, passed, detail, status_if_false="FAIL"):
    rows.append(dict(Category=category, Item=item, Status="PASS" if passed else status_if_false, Detail=str(detail)))


def require_file(relative, category="REQUIRED", minimum_bytes=1):
    path = ROOT / relative
    passed = path.exists() and path.is_file() and path.stat().st_size >= minimum_bytes
    detail = f"{path.stat().st_size} bytes" if path.exists() and path.is_file() else "not present"
    record(category, relative, passed, detail)
    return path if passed else None

required_files = [
    "README.md",
    "AI_USE_DISCLOSURE.md",
    "notebooks/01_QuantumEdge_Reproducibility.ipynb",
    "notebooks/02_QuantumEdge_Graphical_Results.ipynb",
    "notebooks/03_QuantumEdge_Package_Verification.ipynb",
    "notebooks/04_QuantumEdge_MNIST_QRC_Benchmark.ipynb",
    "paper/QuantumEdge_GIC2026_Phase3_Updated_Paper.pdf",
    "data/market_data.csv",
    "data/oxman_spx.csv",
    "requirements.txt",
    "environment.yml",
    "qbraid_skill/quantumedge_phase3/SKILL.md",
    "qbraid_skill/quantumedge_phase3/scripts/run_quantumedge.py",
]
for path in required_files:
    require_file(path)

result_files = [
    "results/headline_metrics.csv",
    "results/oxford_man_metrics.csv",
    "results/ablation.csv",
    "results/reservoir_scaling.csv",
    "results/encoding_density.csv",
    "results/shot_budget.csv",
    "results/noise_zne.csv",
    "results/amplitude_damping.csv",
    "results/forecast_predictions.csv",
    "results/forecast_significance.csv",
    "results/transition_metrics.csv",
    "results/hardware_validation.json",
    "results/hardware_observables.csv",
    "results/mnist_qrc_metrics.csv",
    "results/mnist_manifest.json",
    "results/final_audit.csv",
    "results/run_manifest.json",
]
for path in result_files:
    require_file(path, category="RESULT")

# MNIST authenticity and qubit coverage.
manifest_path = ROOT / "results/mnist_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    record("MNIST", "Canonical MNIST dataset", manifest.get("canonical_mnist") is True, manifest.get("data_source"))
else:
    record("MNIST", "Canonical MNIST dataset", False, "manifest missing")
metrics_path = ROOT / "results/mnist_qrc_metrics.csv"
if metrics_path.exists():
    mnist = pd.read_csv(metrics_path)
    qrc_qubits = set(mnist.loc[mnist["qubits"] > 0, "qubits"].astype(int))
    record("MNIST", "Qubit coverage 5/10/15", {5, 10, 15}.issubset(qrc_qubits), sorted(qrc_qubits))
    record("MNIST", "Accuracy and macro-F1 present", {"accuracy", "macro_f1"}.issubset(mnist.columns) and mnist[["accuracy", "macro_f1"]].notna().all().all(), list(mnist.columns))
else:
    record("MNIST", "Qubit coverage 5/10/15", False, "metrics missing")

# README and qBraid launch button.
readme_path = ROOT / "README.md"
if readme_path.exists():
    readme = readme_path.read_text(encoding="utf-8")
    required_headings = ["Team", "Project", "Challenge track", "Setup", "Run on qBraid", "Inputs", "Outputs", "Limitations"]
    missing = [heading for heading in required_headings if heading.lower() not in readme.lower()]
    record("README", "Required content", not missing, f"missing={missing}")
    has_launch = "account.qbraid.com?gitHubUrl=" in readme and "REPLACE_WITH_PUBLIC_GITHUB_REPO" not in readme
    record("README", "Working Launch on qBraid button", has_launch, "public repository URL required")

# Cover page: only an official template-derived file is accepted by this local check.
cover_candidates = list((ROOT / "paper").glob("*Official*Cover*.pdf")) + list((ROOT / "paper").glob("*Official*Cover*.docx"))
record("COVER", "Official GIC cover page included", bool(cover_candidates), [str(p) for p in cover_candidates] or "not present")

# Paper: 5-page body cannot be counted reliably without PDF parser, but required phrases and wall time can be checked.
paper_text_candidates = [ROOT / "paper/QuantumEdge_GIC2026_Phase3_Updated_Paper.txt"]
for paper_text_path in paper_text_candidates:
    if paper_text_path.exists():
        text = paper_text_path.read_text(encoding="utf-8").lower()
        record("PAPER", "Fez wall-clock reported", "211" in text and "wall" in text, "expect approx 211 seconds")
        record("PAPER", "Marrakesh wall-clock reported", "488" in text and "wall" in text, "expect 488.36 seconds")
        record("PAPER", "MNIST benchmark disclosed", "mnist" in text, "mandatory common benchmark")
        record("PAPER", "Amplitude damping disclosed", "amplitude damping" in text or "amplitude-damping" in text, "noise requirement")
        record("PAPER", "Transition analysis disclosed", "transition" in text and "+/-3" in text, "event-focused evaluation")
        record("PAPER", "Statistical analysis disclosed", "diebold-mariano" in text and "bootstrap" in text, "forecast uncertainty")

# Hardware evidence consistency.
hw_json = ROOT / "results/hardware_validation.json"
hw_csv = ROOT / "results/hardware_observables.csv"
if hw_json.exists() and hw_csv.exists():
    summary = json.loads(hw_json.read_text(encoding="utf-8"))
    points = pd.read_csv(hw_csv)
    corr = float(np.corrcoef(points["simulator"], points["hardware"])[0, 1])
    mae = float(np.mean(np.abs(points["simulator"] - points["hardware"])))
    record("HARDWARE", "39 observable pairs", len(points) == 39, len(points))
    record("HARDWARE", "Correlation recalculates", np.isclose(corr, summary.get("feature_correlation"), atol=1e-10), f"csv={corr}; json={summary.get('feature_correlation')}")
    record("HARDWARE", "MAE recalculates", np.isclose(mae, summary.get("feature_mae"), atol=1e-10), f"csv={mae}; json={summary.get('feature_mae')}")
    record("HARDWARE", "Wall-clock present", float(summary.get("wall_seconds", 0)) > 0, summary.get("wall_seconds"))

# Locked numerical checks.
headline = ROOT / "results/headline_metrics.csv"
if headline.exists():
    table = pd.read_csv(headline, index_col=0)
    refs = {
        ("QRC Dual+Pauli+FB (calibrated)", "RMSE"): (7.532e-5, 0.02),
        ("QRC Dual+Pauli+FB (calibrated)", "QLIKE"): (0.3261, 0.04),
        ("QRC Dual+Pauli+FB", "RMSE"): (7.725e-5, 0.02),
        ("HAR-RV", "RMSE"): (8.005e-5, 0.02),
    }
    for (model, metric), (reference, tolerance) in refs.items():
        try:
            actual = float(table.loc[model, metric])
            passed = np.isclose(actual, reference, rtol=tolerance)
            record("NUMERICAL", f"{model} {metric}", passed, f"actual={actual}; reference={reference}; rtol={tolerance}")
        except Exception as exc:
            record("NUMERICAL", f"{model} {metric}", False, exc)

# Credential scan over text-like files and notebook source/outputs.
secret_patterns = [
    re.compile(r"(?i)(api[_ -]?key|token|password|client[_ -]?secret)\s*[:=]\s*['\"][A-Za-z0-9_\-]{16,}"),
    re.compile(r"(?i)crn:v1:[A-Za-z0-9:_\-/]+"),
]
findings = []
for path in ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in {".py", ".md", ".txt", ".json", ".csv", ".ipynb", ".yml", ".yaml"}:
        continue
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    for pattern in secret_patterns:
        if pattern.search(text):
            findings.append(str(path.relative_to(ROOT)))
record("SECURITY", "Embedded credential scan", not findings, findings or "no likely embedded credentials found")

verification = pd.DataFrame(rows)
Path("results").mkdir(exist_ok=True)
verification.to_csv("results/package_verification.csv", index=False)
summary = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "pass": int((verification.Status == "PASS").sum()),
    "warn": int((verification.Status == "WARN").sum()),
    "fail": int((verification.Status == "FAIL").sum()),
    "report": "results/package_verification.csv",
}
Path("results/package_verification_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

display(verification)
print(json.dumps(summary, indent=2))
if summary["fail"]:
    raise RuntimeError(f"Submission verification has {summary['fail']} failure(s). Resolve them before freezing the ZIP.")
print("Package verification passed without failures.")


,Category,Item,Status,Detail
0,REQUIRED,README.md,PASS,3493 bytes
1,REQUIRED,AI_USE_DISCLOSURE.md,PASS,799 bytes
2,REQUIRED,notebooks/01_QuantumEdge_Reproducibility.ipynb,PASS,109592 bytes
3,REQUIRED,notebooks/02_QuantumEdge_Graphical_Results.ipynb,PASS,42930 bytes
4,REQUIRED,notebooks/03_QuantumEdge_Package_Verification....,PASS,12417 bytes
5,REQUIRED,notebooks/04_QuantumEdge_MNIST_QRC_Benchmark.i...,PASS,20236 bytes
6,REQUIRED,paper/QuantumEdge_GIC2026_Phase3_Updated_Paper...,PASS,596764 bytes
7,REQUIRED,data/market_data.csv,PASS,256024 bytes
8,REQUIRED,data/oxman_spx.csv,PASS,49388 bytes
9,REQUIRED,requirements.txt,PASS,220 bytes


{
  "generated_utc": "2026-07-17T03:18:31.418572+00:00",
  "python": "3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]",
  "pass": 44,
  "warn": 0,
  "fail": 1,
  "report": "results/package_verification.csv"
}


RuntimeError: Submission verification has 1 failure(s). Resolve them before freezing the ZIP.

## Interpretation

- `PASS` means the evidence exists and the local check succeeded.
- `FAIL` means a mandatory submission or scientific artifact is absent or inconsistent.
- The official cover check is deliberately conservative because the organizers prohibit recreating the template.
- The Launch on qBraid test rejects placeholder repository URLs.
- A sklearn digits smoke test never satisfies the MNIST requirement.
